# Query-vocabulary ablation (Scientific Reports, round 3, reviewer point R1.4)

Aurora labels every paper by matching expert-written Boolean queries against its
title, abstract and keywords, and the pool, validation and test papers are all
labelled this way. A method that learns from pool labels could therefore win
partly by recovering the queries' vocabulary rather than the goal. This notebook
tests that directly. **CPU is enough; no LLM is run.** It is modelled on
`04_revision_analyses.ipynb` and uses its data loading, scoring, neighbour vote,
mixing and supervised-baseline code unchanged.

Every term quoted in Aurora's version-5.0 queries (1,034 unique terms, wildcards
included; `artifacts/aurora_queries_v5/`, CC BY 4.0) is removed from the pool,
validation and test texts, and the label-learning methods are rerun on the
masked texts: retrieval-only, the neighbour correction for each backbone, and the
TF-IDF and SBERT classifiers trained on the pool. As a control, the same number
of words is removed from each text at random among the words that are *not*
query terms (`QA_SEEDS` seeds). The LLM predictions stay as they are, so the
LLM-only baselines are unchanged. If a method's gain over the LLM collapses when
the query terms are removed but not when the same amount of other text is
removed, the gain depends on Aurora's query vocabulary.

**Nothing that exists is written to.** All output goes to a new folder per run,
`<root>/aurora_sdg_graph_full/cross_model_query_ablation/run_<date>_<time>/`:
`query_ablation.csv` (every condition, method and backbone),
`query_ablation_summary.csv` (gain over the LLM and the share of it retained),
`query_ablation_masking.csv` (how much text the masking removes) and the SBERT
embeddings of the masked texts. Existing files (caches, earlier results) are
only read; the last cell verifies that none changed.

**Where the inputs are found:** exactly as in notebook 04 (`$SDG_ROOT`, else
config.yaml, else Drive on Colab, with a search of your Drive if needed). The
unmasked condition is checked against the paper's numbers before anything is
reported.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
import os, sys

def _resolve_sdg_root():
    r = os.environ.get('SDG_ROOT')
    if r:
        return os.path.abspath(os.path.expanduser(r))
    for cand in ('config.yaml', os.path.join('..', 'config.yaml')):
        if os.path.exists(cand):
            with open(cand) as fh:
                for line in fh:
                    line = line.split('#')[0].strip()
                    if line.startswith('root:'):
                        v = line.split(':', 1)[1].strip().strip('"\'')
                        if v:
                            return os.path.abspath(os.path.expanduser(v))
    if 'google.colab' in sys.modules or os.path.isdir('/content'):
        try:
            from google.colab import drive
            if not os.path.exists('/content/drive/MyDrive'):
                drive.mount('/content/drive')
            return '/content/drive/MyDrive/sdg-llm-graph'
        except Exception:
            pass
    return os.path.abspath('./sdg_data')

RUN_DIR  = 'aurora_sdg_graph_full'                      # layout of notebooks 02/03
ABS_REL  = os.path.join(RUN_DIR, 'data_cache', 'abstracts_full.parquet')

def _search(top, max_depth=5):
    """Folders under `top` (depth-limited) holding a notebook-02/03 run
    (<dir>/aurora_sdg_graph_full/data_cache/abstracts_full.parquet), and
    released artifacts folders (<dir>/splits/test_labels.npz)."""
    roots, arts = [], []
    top = os.path.abspath(top)
    base = top.rstrip(os.sep).count(os.sep)
    for dirpath, dirnames, _ in os.walk(top):
        if dirpath.count(os.sep) - base >= max_depth:
            dirnames[:] = []
        dirnames[:] = [d for d in dirnames if not d.startswith('.')
                       and d not in ('node_modules', '__pycache__')]
        if RUN_DIR in dirnames and os.path.exists(os.path.join(dirpath, ABS_REL)):
            roots.append(dirpath)
        if 'splits' in dirnames and os.path.exists(
                os.path.join(dirpath, 'splits', 'test_labels.npz')):
            arts.append(dirpath)
    return roots, arts

SDG_ROOT = _resolve_sdg_root()                 # mounts Drive on Colab
ON_COLAB = os.path.isdir('/content/drive/MyDrive')   # so test only after mounting
_explicit = bool(os.environ.get('SDG_ROOT'))

def _artifacts_near(root):
    cands = [os.environ.get('SDG_ARTIFACTS'), os.path.join(root, 'artifacts'),
             os.path.join('..', 'artifacts'), 'artifacts']
    for c in cands:
        if c and os.path.exists(os.path.join(c, 'splits', 'test_labels.npz')):
            return os.path.abspath(c)
    return None

ARTIFACTS = _artifacts_near(SDG_ROOT)

if not os.path.exists(os.path.join(SDG_ROOT, ABS_REL)) and ARTIFACTS is None:
    if _explicit:
        raise FileNotFoundError(
            f'SDG_ROOT={SDG_ROOT} holds no {ABS_REL} and no artifacts/ folder was '
            'found. Point SDG_ROOT at the folder that contains aurora_sdg_graph_full/.')
    top = '/content/drive/MyDrive' if ON_COLAB else os.path.abspath('..')
    print(f'Inputs not under {SDG_ROOT}; searching {top} (this can take a minute on Drive)...')
    roots, arts = _search(top)
    for r in roots: print(f'  found run data : {r}')
    for a in arts:  print(f'  found artifacts: {a}')
    # prefer a root whose run also has the cached predictions of notebook 02
    def _has_preds(r):
        return os.path.isdir(os.path.join(r, RUN_DIR, 'results_v5_multi_gemma4-26b_n10000'))
    ranked = sorted(roots, key=lambda r: (not _has_preds(r), len(r)))
    if len(ranked) > 1 and _has_preds(ranked[0]) == _has_preds(ranked[1]):
        raise RuntimeError('Several candidate data roots were found (listed above). '
                           'Set SDG_ROOT to the one you want, in a cell above this one:\n'
                           "  import os; os.environ['SDG_ROOT'] = '<folder>'")
    if ranked:
        SDG_ROOT = ranked[0]
    elif arts:
        ARTIFACTS = arts[0]
        SDG_ROOT = os.path.dirname(arts[0])
    else:
        raise FileNotFoundError(
            f'No run data ({ABS_REL}) and no artifacts/ folder found under {top}. '
            "Set SDG_ROOT to the folder holding aurora_sdg_graph_full/, or "
            'SDG_ARTIFACTS to the repository\'s artifacts/ folder.')
    ARTIFACTS = ARTIFACTS or _artifacts_near(SDG_ROOT)

DRIVE_ROOT = os.path.join(SDG_ROOT, RUN_DIR)
DATA_CACHE = os.path.join(DRIVE_ROOT, 'data_cache')
assert os.path.isdir(DATA_CACHE), f'data cache not found: {DATA_CACHE}'   # read only
# New output folder for this run; nothing that exists is written to.
import time
QA_OUT = os.path.join(DRIVE_ROOT, 'cross_model_query_ablation', time.strftime('run_%Y%m%d_%H%M%S'))
os.makedirs(QA_OUT)          # fails rather than reuse an existing folder

# Input folders, in order of preference: the run's own cache, then the release.
_cache_dirs = [DATA_CACHE] + ([os.path.join(ARTIFACTS, 'data_cache')] if ARTIFACTS else [])

def _locate(fname, subdirs):
    """First existing <dir>/<fname> among the given directories, else None."""
    for d in subdirs:
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    return None

print(f'SDG_ROOT   : {SDG_ROOT}')
print(f'DATA_CACHE : {DATA_CACHE}')
print(f'ARTIFACTS  : {ARTIFACTS or "(not found - not needed)"}')
print(f'QA_OUT     : {QA_OUT}')

In [ ]:
import importlib.util, subprocess
for _pkg, _mod in [('pyarrow', 'pyarrow'), ('scikit-learn', 'sklearn'),
                   ('scipy', 'scipy'), ('pandas', 'pandas')]:
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=True)

import json, hashlib, time, re
import numpy as np, pandas as pd
from scipy import stats

# ---- CONFIG -----------------------------------------------------------------
SEED       = 42
MODELS     = ['gemma4-26b', 'mistral-24b', 'qwen3-32b', 'mixtral-8x7b']
PRIMARY    = 'gemma4-26b'
K_HEAD     = 50
CANON_GRID = np.arange(0.10, 0.851, 0.05)     # paper's threshold grid (16 points)
LAM_GRID   = np.arange(0.0, 1.01, 0.1)        # identical to notebook 03, Cell U
N_BOOT     = 1000
BOOT_SEED  = 12345                            # identical to notebook 03, Cell U
SBERT_NAME = 'all-MiniLM-L6-v2'

N_CONTROL_SEEDS = 20                          # A2: shuffled labels, random neighbours
SIM_BANDS  = [(0, 50), (50, 100), (100, 200), (200, 400), (400, 800),
              (1000, 1050), ('bottom', 50)]   # A2: neighbour rank bands
POOL_SIZES = [50, 100, 250, 500, 1000, 2000]  # A3 (full pool 3054 added automatically)
N_POOL_SEEDS = 10
LOGREG_C   = [0.1, 1.0, 10.0]                 # B1: selected on validation

# Split fingerprints (SHA-1 of the ordered DOI list, first 12 hex), as in the paper.
FINGERPRINT = {'val': 'b212c22b4951', 'test': '4c5581303537', 'pool': 'c8ffa1517ecc'}

# The paper's numbers (cross_model_comparison_v6, written by notebook 03 Cell U).
# The gate below must reproduce them; if it fails, results are not comparable.
V6_REF = {
    'gemma4-26b':   {'B1': 0.5216177784976842, 'B2': 0.5286151349954628,
                     'doc_neighbours': 0.611613262299146,  'lambda': 0.7},
    'mistral-24b':  {'B1': 0.4787128317504444, 'B2': 0.4589641221985147,
                     'doc_neighbours': 0.5940953946602423, 'lambda': 0.8},
    'qwen3-32b':    {'B1': 0.4240813821399711, 'B2': 0.4024134188787943,
                     'doc_neighbours': 0.5831047725927838, 'lambda': 0.7},
    'mixtral-8x7b': {'B1': 0.2198128864316267, 'B2': 0.1829298028348064,
                     'doc_neighbours': 0.5821240490870296, 'lambda': 1.0},
}
V6_RETRIEVAL_ONLY = 0.5821240490870296
V6_BOOT_NBR_B2    = 0.0829776942864933        # gemma, doc_neighbours - B2, mean

STRICT_REPRO_GATE = os.environ.get('SDG_STRICT_GATE', '1') != '0'
REPRO_TOL = 1e-6

## 1. Splits, labels, texts, predictions
Same sources and slicing as notebook 03: the abstract cache sliced
sequentially (few-shot 40, val 200, test 10,000) and the neighbour pool cache
filtered at 50 characters. Labels are the parquet's goal columns. Every split
is checked against its published fingerprint.

In [ ]:
def _fp(dois):
    return hashlib.sha1(''.join(dois).encode('utf-8')).hexdigest()[:12]

p_abs  = _locate('abstracts_full.parquet', _cache_dirs)
p_pool = _locate(f'neighbour_pool_abstracts_n5000_seed{SEED}.parquet', _cache_dirs)
assert p_abs,  f'abstracts_full.parquet not found in {_cache_dirs}'
assert p_pool, f'neighbour_pool_abstracts_n5000_seed{SEED}.parquet not found in {_cache_dirs}'
print(f'abstracts : {p_abs}\npool      : {p_pool}')

df_all  = pd.read_parquet(p_abs)
df_val  = df_all.iloc[40:240].reset_index(drop=True)
df_test = df_all.iloc[240:10240].reset_index(drop=True)
df_pool = pd.read_parquet(p_pool)
df_pool = df_pool[df_pool['abstract'].fillna('').str.len() > 50].reset_index(drop=True)

for name, df in [('val', df_val), ('test', df_test), ('pool', df_pool)]:
    fp = _fp(df['doi'].astype(str).tolist())
    assert fp == FINGERPRINT[name], \
        f'{name}: fingerprint {fp} != published {FINGERPRINT[name]} (different cache?)'
    print(f'  {name:5s} n={len(df):5d}  fingerprint {fp}  OK')

goal_cols = sorted([c for c in df_all.columns if re.fullmatch(r'sdg_\d+', c)],
                   key=lambda c: int(c.split('_')[1]))
assert len(goal_cols) == 17
y_val  = df_val[goal_cols].values.astype(int)
y_test = df_test[goal_cols].values.astype(int)
y_pool = df_pool[goal_cols].values.astype(np.float32)

# Cross-check against the released label arrays when the artifacts are around.
if ARTIFACTS:
    for name, y in [('val', y_val), ('test', y_test), ('pool', y_pool)]:
        ref = np.load(os.path.join(ARTIFACTS, 'splits', f'{name}_labels.npz'))['goal']
        assert np.array_equal(y, ref), f'{name}: labels differ from artifacts/splits'
    print('  labels identical to artifacts/splits')

def _text(df):
    return (df['title'].fillna('') + '. ' + df['abstract'].fillna('')).tolist()

txt_val, txt_test, txt_pool = _text(df_val), _text(df_test), _text(df_pool)
has_abstract = (df_test['abstract'].fillna('').str.len() > 50).values

# Raw LLM predictions: notebook 02's result folders, else the released .npz.
cached = {}
for m in MODELS:
    tag = f'v5_multi_{m}_n10000'
    rd = os.path.join(DRIVE_ROOT, f'results_{tag}')
    files = {f'{k}_{s}': os.path.join(rd, f'{k}_probs_{s}_{tag}.npy')
             for k in ('goal', 'target') for s in ('val', 'test')}
    npz = (_locate(f'raw_llm_predictions_{m}_n10000.npz',
                   [os.path.join(ARTIFACTS, 'predictions')]) if ARTIFACTS else None)
    if all(os.path.exists(f) for f in files.values()):
        cached[m] = {k: np.load(f) for k, f in files.items()}
        src = rd
    elif npz:
        z = np.load(npz)
        cached[m] = {f'{k}_{s}': z[f'{k}_probs_{s}_{tag}']
                     for k in ('goal', 'target') for s in ('val', 'test')}
        src = npz
    else:
        raise FileNotFoundError(f'{m}: no cached predictions in {rd} and no released .npz')
    assert cached[m]['goal_test'].shape == (10000, 17)
    print(f'  {m:13s} predictions from {src}')

## 2. SBERT embeddings (computed once, cached under DATA_CACHE)

In [ ]:
# Keyed by model name and the three split fingerprints. A copy shipped in
# artifacts/data_cache is used if present, so the neighbour vote does not depend
# on the locally installed sentence-transformers / torch versions.
_EMB_NAME = (f'sbert_{SBERT_NAME}_val{FINGERPRINT["val"]}'
             f'_test{FINGERPRINT["test"]}_pool{FINGERPRINT["pool"]}.npz')
EMB_CACHE = _locate(_EMB_NAME, _cache_dirs) or os.path.join(DATA_CACHE, _EMB_NAME)

if os.path.exists(EMB_CACHE):
    z = np.load(EMB_CACHE)
    val_emb, test_emb, pool_emb = z['val'], z['test'], z['pool']
    print(f'Embeddings loaded from cache: {EMB_CACHE}')
else:
    if importlib.util.find_spec('sentence_transformers') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'sentence-transformers'], check=True)
    from sentence_transformers import SentenceTransformer
    sbert = SentenceTransformer(SBERT_NAME)
    t0 = time.time()
    enc = lambda t: sbert.encode(t, normalize_embeddings=True, batch_size=64,
                                 show_progress_bar=False).astype(np.float32)
    pool_emb, val_emb, test_emb = enc(txt_pool), enc(txt_val), enc(txt_test)
    # 05: not cached in the data folder; saved to this run's folder instead.
    np.savez_compressed(os.path.join(QA_OUT, os.path.basename(EMB_CACHE)),
                        val=val_emb, test=test_emb, pool=pool_emb)
    print(f'Encoded in {time.time()-t0:.0f}s, saved to {QA_OUT}')

print(f'pool {pool_emb.shape}  val {val_emb.shape}  test {test_emb.shape}')

## 3. Scoring, neighbour vote, mixing (verbatim from notebook 03, Cells U/R)

In [ ]:
TARGETS_PER_GOAL = [7,8,13,10,9,8,5,12,8,10,10,11,5,10,12,12,19]
GOAL_TO_TARGETS, _i = {}, 0
for g, n in enumerate(TARGETS_PER_GOAL, start=1):
    GOAL_TO_TARGETS[g] = list(range(_i, _i + n)); _i += n

def aggregate_targets_to_goals(target_probs):
    out = np.zeros((target_probs.shape[0], 17), dtype=np.float32)
    for g in range(1, 18):
        out[:, g-1] = target_probs[:, GOAL_TO_TARGETS[g]].max(axis=1)
    return out

def _f1_bin(yt, yp):
    tp = float(np.count_nonzero(yt & yp))
    fp = float(np.count_nonzero((~yt) & yp))
    fn = float(np.count_nonzero(yt & (~yp)))
    d = 2*tp + fp + fn
    return 0.0 if d == 0 else 2*tp/d

def fit_thresholds(probs, labels, grid=CANON_GRID):
    yb = labels.astype(bool)
    thr = np.zeros(probs.shape[1])
    for c in range(probs.shape[1]):
        col, yc = probs[:, c], yb[:, c]
        best_f, best_t = -1.0, float(grid[0])
        for t in grid:
            f = _f1_bin(yc, col >= t)
            if f > best_f:
                best_f, best_t = f, float(t)
        thr[c] = best_t
    return thr

def per_class_f1(probs, labels, thr):
    yb, pred = labels.astype(bool), probs >= thr
    return np.array([_f1_bin(yb[:, c], pred[:, c]) for c in range(probs.shape[1])])

def score(probs, labels, thr):
    return float(per_class_f1(probs, labels, thr).mean())

def fit_and_score(p_val, p_test):
    thr = fit_thresholds(p_val, y_val)
    return score(p_val, y_val, thr), score(p_test, y_test, thr), thr

def vote_from_idx(sims, idx, labels):
    """Eq. 5: similarity-weighted vote over the neighbour index set idx (n x K)."""
    out = np.zeros((idx.shape[0], labels.shape[1]), dtype=np.float64)
    for i in range(idx.shape[0]):
        nn = idx[i]
        w = np.clip(sims[i, nn], 0, None); Z = w.sum()
        out[i] = (w @ labels[nn]) / Z if Z > 0 else labels[nn].mean(0)
    return out

def nbr_vote(q_emb, p_emb, p_lab, K):
    """Top-K vote, exactly as Cell U's nbr_vote(weighted=True)."""
    sims = q_emb @ p_emb.T
    idx = np.argpartition(-sims, K-1, axis=1)[:, :K]
    return vote_from_idx(sims, idx, p_lab)

def calibrate_mix(gv, gt, ev, et):
    """Global lambda on validation (first maximum), as Cell U. Returns dict."""
    best = None
    for lam in LAM_GRID:
        v, t, thr = fit_and_score((1-lam)*gv + lam*ev, (1-lam)*gt + lam*et)
        if best is None or v > best['val']:
            best = {'lam': float(lam), 'val': v, 'test': t, 'thr': thr,
                    'p_test': (1-lam)*gt + lam*et}
    return best

def boot_diff(pA, thrA, pB, thrB, mask=None, n_boot=N_BOOT, seed=BOOT_SEED):
    """Paired document-level bootstrap of macro-F1(A) - macro-F1(B), Cell U."""
    yt = y_test if mask is None else y_test[mask]
    pA = pA if mask is None else pA[mask]
    pB = pB if mask is None else pB[mask]
    rs = np.random.RandomState(seed); n = pA.shape[0]; d = np.empty(n_boot)
    for k in range(n_boot):
        b = rs.randint(0, n, n)
        d[k] = score(pA[b], yt[b], thrA) - score(pB[b], yt[b], thrB)
    return float(d.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

def mcnemar(pA, thrA, pB, thrB):
    """Per-(paper, goal) correctness. b: A right, B wrong; c: the converse."""
    okA = (pA >= thrA).astype(int) == y_test
    okB = (pB >= thrB).astype(int) == y_test
    b, c = int((okA & ~okB).sum()), int((~okA & okB).sum())
    if b + c >= 25:
        p = float(stats.chi2.sf((abs(b - c) - 1) ** 2 / (b + c), 1))
    else:
        p = float(stats.binomtest(b, b + c, 0.5).pvalue) if b + c else 1.0
    return b, c, p

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def ovr_logreg(Xtr, ytr, Xv, Xt, C):
    """One-vs-rest, class-balanced logistic regression; a goal with a single
    class in the training labels gets its constant training frequency."""
    pv = np.zeros((Xv.shape[0], 17)); pt = np.zeros((Xt.shape[0], 17))
    for g in range(17):
        yg = ytr[:, g].astype(int)
        if yg.min() == yg.max():
            pv[:, g] = pt[:, g] = float(yg[0]); continue
        clf = LogisticRegression(C=C, class_weight='balanced', max_iter=2000,
                                 random_state=SEED)
        clf.fit(Xtr, yg)
        pv[:, g] = clf.predict_proba(Xv)[:, 1]; pt[:, g] = clf.predict_proba(Xt)[:, 1]
    return pv, pt

def supervised(features, idx):
    """Train on pool rows idx; C chosen on validation (first maximum)."""
    if features == 'TF-IDF':
        vec = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.9,
                              ngram_range=(1, 2), stop_words='english',
                              max_features=50000)
        Xtr = vec.fit_transform([txt_pool[i] for i in idx])
        Xv, Xt = vec.transform(txt_val), vec.transform(txt_test)
    else:
        Xtr, Xv, Xt = pool_emb[idx], val_emb, test_emb
    best = None
    for C in LOGREG_C:
        pv, pt = ovr_logreg(Xtr, y_pool[idx], Xv, Xt, C)
        v, t, thr = fit_and_score(pv, pt)
        if best is None or v > best['val']:
            best = {'val': v, 'test': t, 'thr': thr, 'C': C, 'pv': pv, 'pt': pt}
    return best

# Shared quantities used by every part below.
ev_val  = nbr_vote(val_emb,  pool_emb, y_pool, K_HEAD)
ev_test = nbr_vote(test_emb, pool_emb, y_pool, K_HEAD)
SIMS_TEST = test_emb @ pool_emb.T
SIMS_VAL  = val_emb  @ pool_emb.T

BASE = {}
for m in MODELS:
    gv = cached[m]['goal_val'].astype(np.float64); gt = cached[m]['goal_test'].astype(np.float64)
    av = aggregate_targets_to_goals(cached[m]['target_val'])
    at = aggregate_targets_to_goals(cached[m]['target_test'])
    b1v, b1t, thr1 = fit_and_score(gv, gt)
    b2v, b2t, thr2 = fit_and_score(av, at)
    BASE[m] = dict(gv=gv, gt=gt, at=at, b1=b1t, b2=b2t, thr1=thr1, thr2=thr2,
                   mix=calibrate_mix(gv, gt, ev_val, ev_test))
rv, rt, thr_r = fit_and_score(ev_val, ev_test)
RETR = dict(val=rv, test=rt, thr=thr_r)

## C. Query-vocabulary ablation

In [ ]:
# ── Record every existing file under the data folder (verified at the end) ───
def _snapshot(top=DRIVE_ROOT):
    snap = {}
    for dirpath, dirnames, files in os.walk(top):
        if os.path.abspath(dirpath) == os.path.abspath(QA_OUT):
            dirnames[:] = []; continue
        for fn in files:
            q = os.path.join(dirpath, fn)
            try:
                st = os.stat(q); snap[q] = (st.st_size, st.st_mtime)
            except OSError:
                pass
    return snap
_BEFORE = _snapshot()
print(f'Existing files recorded: {len(_BEFORE)}')

In [ ]:
# ── C1. Aurora query terms and masking ──────────────────────────────────────
import urllib.request
QA_COMMIT = '9acd03d00b6c54cede19f5eb0b45eec193e5724b'   # Aurora sdg-queries, v5.0 queries
QA_SEEDS  = 3                                            # random-word control seeds

def _query_xml(g):
    """Shipped copy (artifacts/aurora_queries_v5) if present, else the same
    commit from GitHub, read into memory (nothing is written)."""
    fn = f'query_SDG{g}.xml'
    if ARTIFACTS and os.path.exists(os.path.join(ARTIFACTS, 'aurora_queries_v5', fn)):
        with open(os.path.join(ARTIFACTS, 'aurora_queries_v5', fn), encoding='utf-8-sig') as f:
            return f.read(), 'artifacts/aurora_queries_v5'
    url = (f'https://raw.githubusercontent.com/Aurora-Network-Global/sdg-queries/'
           f'{QA_COMMIT}/{fn}')
    return urllib.request.urlopen(url, timeout=60).read().decode('utf-8-sig'), url

QA_TERMS, _src = set(), set()
for g in range(1, 18):
    xml, src = _query_xml(g); _src.add(src.split('/query_SDG')[0])
    for line in re.findall(r'<aqd:query-line field="([^"]*)">(.*?)</aqd:query-line>', xml, re.S):
        assert 'TITLE' in line[0] or 'ABS' in line[0], line[0]
        QA_TERMS.update(t.strip().lower() for t in re.findall(r'"([^"]+)"', line[1]) if t.strip())
print(f'Aurora v5.0 queries from {sorted(_src)}: {len(QA_TERMS)} unique quoted terms')

def _term_pattern(t):
    """Scopus term -> regex: '*' any word ending, '?' one optional character,
    words separated by spaces or hyphens; whole words only."""
    words = [re.escape(w).replace(r'\*', r'\w*').replace(r'\?', r'\w?')
             for w in re.split(r'[\s]+', t)]
    return r'[\s\-]+'.join(words) + r'(?!\w)'

QA_RX = re.compile(r'(?<!\w)(?:' + '|'.join(sorted((_term_pattern(t) for t in QA_TERMS),
                                                  key=len, reverse=True)) + ')', re.I)
_WORD = re.compile(r'\w+')

def _query_spans(text):
    return [m.span() for m in QA_RX.finditer(text)]

def _drop_spans(text, spans):
    out, last = [], 0
    for a, b in sorted(spans):
        if a < last: continue
        out.append(text[last:a]); last = b
    out.append(text[last:])
    return re.sub(r'\s+', ' ', ' '.join(out)).strip()

def mask_query(texts):
    """Remove every Aurora query-term occurrence; return texts and the number
    of words removed from each."""
    out, n_rm, n_tok = [], [], []
    for t in texts:
        sp = _query_spans(t)
        n_rm.append(sum(len(_WORD.findall(t[a:b])) for a, b in sp))
        n_tok.append(len(_WORD.findall(t)))
        out.append(_drop_spans(t, sp))
    return out, np.array(n_rm), np.array(n_tok)

def mask_random(texts, n_rm, seed):
    """Control: remove the same number of words from each text, drawn at random
    among the words that are not inside a query-term match."""
    rs, out = np.random.RandomState(seed), []
    for t, k in zip(texts, n_rm):
        sp = _query_spans(t)
        inside = lambda a, b: any(a < qb and b > qa for qa, qb in sp)
        cand = [m.span() for m in _WORD.finditer(t) if not inside(*m.span())]
        k = min(int(k), len(cand))
        pick = [cand[i] for i in rs.choice(len(cand), k, replace=False)] if k else []
        out.append(_drop_spans(t, pick))
    return out

# Masked texts for the three splits.
qm_pool, rm_pool, tok_pool = mask_query(txt_pool)
qm_val,  rm_val,  tok_val  = mask_query(txt_val)
qm_test, rm_test, tok_test = mask_query(txt_test)
_mask_rows = []
for name, rm, tok in [('pool', rm_pool, tok_pool), ('val', rm_val, tok_val), ('test', rm_test, tok_test)]:
    _mask_rows.append({'split': name, 'n_texts': len(rm),
                       'share_with_query_term': float((rm > 0).mean()),
                       'mean_words': float(tok.mean()),
                       'mean_words_removed': float(rm.mean()),
                       'share_of_words_removed': float(rm.sum() / max(tok.sum(), 1))})
df_qa_mask = pd.DataFrame(_mask_rows)
df_qa_mask.to_csv(os.path.join(QA_OUT, 'query_ablation_masking.csv'), index=False)
print(df_qa_mask.round(4).to_string(index=False))
print('\nExample (first test paper):\n  before:', txt_test[0][:300], '\n  after :', qm_test[0][:300])

In [ ]:
# ── C2. Rerun the label-learning methods on each version of the texts ────────
if importlib.util.find_spec('sentence_transformers') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'], check=True)
from sentence_transformers import SentenceTransformer
_qa_sbert = SentenceTransformer(SBERT_NAME)
_qa_enc = lambda t: _qa_sbert.encode(t, normalize_embeddings=True, batch_size=64,
                                     show_progress_bar=False).astype(np.float32)

def _supervised_on(feat, tp, tv, tt, ep, ev, et):
    """As supervised() above, on the given texts / embeddings (full pool)."""
    if feat == 'TF-IDF':
        vec = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.9,
                              ngram_range=(1, 2), stop_words='english',
                              max_features=50000)
        Xtr = vec.fit_transform(tp); Xv, Xt = vec.transform(tv), vec.transform(tt)
    else:
        Xtr, Xv, Xt = ep, ev, et
    best = None
    for C in LOGREG_C:
        pv, pt = ovr_logreg(Xtr, y_pool, Xv, Xt, C)
        v, t, thr = fit_and_score(pv, pt)
        if best is None or v > best['val']:
            best = {'val': v, 'test': t, 'C': C}
    return best

def _run_condition(cond, seed, tp, tv, tt, emb=None):
    t0 = time.time()
    if emb is None:
        ep, ev, et = _qa_enc(tp), _qa_enc(tv), _qa_enc(tt)
        np.savez_compressed(os.path.join(QA_OUT, f'sbert_{cond.replace(" ", "_")}'
                                         f'{"" if seed is None else f"_seed{seed}"}.npz'),
                            pool=ep, val=ev, test=et)
    else:
        ep, ev, et = emb
    vv = nbr_vote(ev, ep, y_pool, K_HEAD); vt = nbr_vote(et, ep, y_pool, K_HEAD)
    rv_, rt_, _ = fit_and_score(vv, vt)
    rows = [{'condition': cond, 'seed': seed, 'method': 'retrieval-only', 'model': '-',
             'val_f1': rv_, 'test_f1': rt_, 'lambda': None, 'C': None}]
    for m in MODELS:
        B = BASE[m]; mx = calibrate_mix(B['gv'], B['gt'], vv, vt)
        rows.append({'condition': cond, 'seed': seed, 'method': 'neighbour correction',
                     'model': m, 'val_f1': mx['val'], 'test_f1': mx['test'],
                     'lambda': mx['lam'], 'C': None})
    for feat in ('TF-IDF', 'SBERT'):
        b = _supervised_on(feat, tp, tv, tt, ep, ev, et)
        rows.append({'condition': cond, 'seed': seed, 'method': f'{feat} + LogReg (pool)',
                     'model': '-', 'val_f1': b['val'], 'test_f1': b['test'],
                     'lambda': None, 'C': b['C']})
    print(f'  {cond:24s} seed={seed}  done in {time.time()-t0:.0f}s')
    return rows

qa_rows = _run_condition('original', None, txt_pool, txt_val, txt_test,
                         emb=(pool_emb, val_emb, test_emb))
# The unmasked condition must reproduce the numbers above and the paper's.
_o = {(r['method'], r['model']): r['test_f1'] for r in qa_rows}
assert abs(_o[('retrieval-only', '-')] - RETR['test']) < 1e-12
for m in MODELS:
    assert abs(_o[('neighbour correction', m)] - BASE[m]['mix']['test']) < 1e-12
_paper_ok = (abs(RETR['test'] - V6_RETRIEVAL_ONLY) <= REPRO_TOL and
             all(abs(BASE[m]['mix']['test'] - V6_REF[m]['doc_neighbours']) <= REPRO_TOL
                 for m in MODELS))
print('  unmasked condition reproduces the paper:', 'YES' if _paper_ok else
      'NO (embeddings differ from the original run; differences below are still internally consistent)')

qa_rows += _run_condition('query terms removed', None, qm_pool, qm_val, qm_test)
for s in range(QA_SEEDS):
    qa_rows += _run_condition('random words removed', s,
                              mask_random(txt_pool, rm_pool, SEED + 2000 + s),
                              mask_random(txt_val,  rm_val,  SEED + 3000 + s),
                              mask_random(txt_test, rm_test, SEED + 4000 + s))

df_qa = pd.DataFrame(qa_rows)
for i, r in df_qa.iterrows():
    ref = BASE[r['model'] if r['model'] != '-' else PRIMARY]['b1']
    df_qa.loc[i, 'minus_B1'] = r['test_f1'] - ref
df_qa.to_csv(os.path.join(QA_OUT, 'query_ablation.csv'), index=False)

In [ ]:
# ── C3. Summary: how much of each method's gain over the LLM survives ────────
# Gain = test macro-F1 minus the LLM-only baseline B1 (for the LLM-free methods,
# B1 of the primary backbone). Retained = gain after masking / gain unmasked.
summ = []
for (meth, model), g in df_qa.groupby(['method', 'model'], sort=False):
    o  = g[g.condition == 'original'].iloc[0]
    q  = g[g.condition == 'query terms removed'].iloc[0]
    rr = g[g.condition == 'random words removed']
    summ.append({'method': meth, 'model': model if model != '-' else f'(B1 of {PRIMARY})',
                 'B1': o['test_f1'] - o['minus_B1'],
                 'original': o['test_f1'],
                 'query_terms_removed': q['test_f1'],
                 'random_words_removed': rr['test_f1'].mean(),
                 'random_sd': rr['test_f1'].std(ddof=1) if len(rr) > 1 else np.nan,
                 'gain_original': o['minus_B1'],
                 'gain_query_removed': q['minus_B1'],
                 'gain_random_removed': rr['minus_B1'].mean(),
                 'retained_query': q['minus_B1'] / o['minus_B1'] if o['minus_B1'] else np.nan,
                 'retained_random': rr['minus_B1'].mean() / o['minus_B1'] if o['minus_B1'] else np.nan})
df_qa_sum = pd.DataFrame(summ)
df_qa_sum.to_csv(os.path.join(QA_OUT, 'query_ablation_summary.csv'), index=False)
pd.set_option('display.width', 200)
print(df_qa_sum.round(4).to_string(index=False))
print(f'\nWritten to {QA_OUT}:')
for f in sorted(os.listdir(QA_OUT)): print('  ', f)

In [ ]:
# ── Nothing that existed before this run was changed ─────────────────────────
_after = _snapshot()
_changed = sorted(f for f in _BEFORE if f in _after and _after[f] != _BEFORE[f])
_gone    = sorted(f for f in _BEFORE if f not in _after)
_new     = sorted(f for f in _after if f not in _BEFORE)
print(f'Existing files checked: {len(_BEFORE)}.  Changed: {len(_changed)}.  Missing: {len(_gone)}.  '
      f'New files outside this run\'s folder: {len(_new)}.')
for f in (_changed + _gone + _new)[:20]: print('   ', f)
assert not (_changed or _gone or _new), 'An existing file was touched - please report this.'
print(f'All outputs of this run are in {QA_OUT}')